# Prótotipo do Modelo de Classificação Binário de Hanseníase

## Treinamento do Zero (sem pesos pré-treinados)

In [1]:
import sys
import os

sys.path.append(os.path.abspath('../..'))

In [7]:
# 1 - Importações
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

In [8]:
# 2 - Importação do Modelo ResNet50 sem pesos pré-treinados
pre_treined_model = tf.keras.applications.ResNet50(weights=None, include_top=False)

2025-08-06 14:15:49.746204: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [9]:
# 3 - Criando as Camadas Densas Personalizadas
x = pre_treined_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
predis = tf.keras.layers.Dense(1, activation='sigmoid')(x)

modelo_binario = tf.keras.Model(inputs=pre_treined_model.input, outputs=predis)

# Tornando todas as camadas treináveis
for layer in modelo_binario.layers:
    layer.trainable = True

In [4]:
# 4 - Preparação do Dataset
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    'train_images_binary',
    target_size=(224, 224),
    color_mode='rgb',
    batch_size=32,
    class_mode='binary',
    shuffle=True,
    subset='training'
)

Found 1098 images belonging to 2 classes.


In [5]:
# 5 - Compilando o Modelo
modelo_binario.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
# 6 - Callbacks
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
# 7 - Treinamento
history = modelo_binario.fit(
    train_generator,
    epochs=40,
    callbacks=[reduce_lr]
)

In [14]:
# 8 - Salvando o Modelo
from utils.models_to_pkl import save_model
save_model(modelo_binario, "modelo_binario_do_zero")

NameError: name 'modelo_binario' is not defined

In [ ]:
# 9 - Avaliação Gráfica
accuracy = history.history["accuracy"]
loss = history.history["loss"]

plt.figure()
plt.plot(accuracy, label="Evolução da Acurácia Durante Treinamento")
plt.xlabel("Epochs")
plt.ylabel("Acurácia")
plt.legend()

plt.figure()
plt.plot(loss, label="Evolução da Perda Durante Treinamento")
plt.xlabel("Epochs")
plt.ylabel("Perda")
plt.legend()